In [1]:
pip install pytesseract

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import pytesseract

In [3]:
img=cv2.imread('card1.png')
img

array([[[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       ...,

       [[255, 255, 255],
        [255, 255, 255],
        [253, 253, 253],
        ...,
        [150, 150, 150],
        [246, 246, 246],
        [250, 250, 250]],

       [[255, 255, 255],
        [255, 255, 255],
        [254, 254, 254],
        ...,
        [241, 241, 241],
        [248, 248, 248],
        [251, 251, 251]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [251, 251, 251],
        [255, 255, 255],
        [255, 255, 255]]

In [4]:
cv2.imshow("card1", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [6]:
data=pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
data

{'level': [1, 2, 3, 4, 5, 2, 3, 4, 5, 5, 2, 3, 4, 5, 5, 2, 3, 4, 5, 5, 5, 5],
 'page_num': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'block_num': [0,
  1,
  1,
  1,
  1,
  2,
  2,
  2,
  2,
  2,
  3,
  3,
  3,
  3,
  3,
  4,
  4,
  4,
  4,
  4,
  4,
  4],
 'par_num': [0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1],
 'line_num': [0,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1],
 'word_num': [0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  2,
  0,
  0,
  0,
  1,
  2,
  0,
  0,
  0,
  1,
  2,
  3,
  4],
 'left': [0,
  0,
  0,
  0,
  0,
  25,
  25,
  25,
  25,
  168,
  25,
  25,
  25,
  25,
  182,
  25,
  25,
  25,
  25,
  71,
  117,
  163],
 'top': [0,
  0,
  0,
  0,
  0,
  17,
  17,
  17,
  17,
  21,
  43,
  43,
  43,
  43,
  47,
  77,
  77,
  77,
  85,
  85,
  85,
  77],
 'width': [220,
  220,
  220,
  220,
  220,
  172,
  172,
  172,
  75,
  2

In [7]:
import re

In [8]:
import re
import cv2
import pytesseract

img = cv2.imread("card1.png")

# Upscaled grayscale (no threshold) + thresholded copy
scale = 2
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
big = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
_, thr = cv2.threshold(big, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

data = pytesseract.image_to_data(thr, output_type=pytesseract.Output.DICT)

date_text = None

for i in range(len(data['text'])):
    token = data['text'][i].strip()

    # loose match: MM/ followed by 2-4 digits (handles OCR noise like 09/248)
    if re.search(r'\b(0[1-9]|1[0-2])\s*/\s*\d{2,4}\b', token):
        x, y = data['left'][i], data['top'][i]
        w, h = data['width'][i], data['height'][i]

        # crop that area (with padding) and re-read digits only
        pad = 10
        crop = big[max(y - pad, 0): y + h + pad, max(x - pad, 0): x + w + pad]
        crop = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

        cfg = "--psm 7 -c tessedit_char_whitelist=0123456789/"
        reread = pytesseract.image_to_string(crop, config=cfg).strip()

        m = re.search(r'(0[1-9]|1[0-2])/(\d{2})\b', reread)
        date_text = m.group(0) if m else token

        # draw box on the ORIGINAL image
        cv2.rectangle(img,
                      (int(x / scale), int(y / scale)),
                      (int((x + w) / scale), int((y + h) / scale)),
                      (0, 0, 255), 2)
        break   # stop after the first date

if date_text:
    print("Date:", date_text)
else:
    print("No date found")

cv2.imshow("IMAGE", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Date: 09/28


In [9]:
import re
import cv2
import pytesseract
import calendar
from datetime import date

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

img = cv2.imread("card1.png")          # fresh copy every run
today = date.today()

# Preprocess
scale = 2
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
big = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
_, thr = cv2.threshold(big, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

data = pytesseract.image_to_data(thr, output_type=pytesseract.Output.DICT)

expiry = None

for i in range(len(data['text'])):
    token = data['text'][i].strip()

    # loose match to LOCATE the date (accepts noisy reads like 09/248)
    if not re.search(r'\b(0[1-9]|1[0-2])\s*/\s*\d{2,4}\b', token):
        continue

    x, y = data['left'][i], data['top'][i]
    w, h = data['width'][i], data['height'][i]

    # re-read just that area, digits and "/" only
    pad = 10
    crop = big[max(y - pad, 0): y + h + pad, max(x - pad, 0): x + w + pad]
    crop = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    cfg = "--psm 7 -c tessedit_char_whitelist=0123456789/"
    reread = pytesseract.image_to_string(crop, config=cfg).strip()

    m = re.search(r'(0[1-9]|1[0-2])/(\d{2})\b', reread)
    if not m:
        continue

    month = int(m.group(1))
    year = 2000 + int(m.group(2))

    # sanity check: expiry should be within a realistic range
    if not (today.year - 10 <= year <= today.year + 15):
        continue

    expiry = date(year, month, calendar.monthrange(year, month)[1])

    cv2.rectangle(img,
                  (int(x / scale), int(y / scale)),
                  (int((x + w) / scale), int((y + h) / scale)),
                  (0, 0, 255), 2)
    break

# Output
if expiry is None:
    print("Could not read a date from the card")
else:
    days_left = (expiry - today).days
    print("Expiry date:", expiry.strftime("%m/%Y"))

    if days_left < 0:
        print("No validity (card expired)")
    elif days_left <= 90:
        print(f"Validity ends soon ({days_left} days left)")
    else:
        print(f"Valid for {days_left} more days")

cv2.imshow("IMAGE", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

Expiry date: 09/2028
Valid for 680 more days
